In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/CRAST/npm_features_final.csv'
df = pd.read_csv(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nClass distribution:")
print(df['label'].value_counts())

df.head()

Shape: (96, 132)

Columns:
['sample_id', 'label', 'trace_file_count', 'total_syscalls', 'unique_syscalls', 'failed_syscalls', 'failed_syscall_ratio', 'top1_syscall_ratio', 'top3_syscall_ratio', 'trace_entropy', 'file_ops_count', 'process_ops_count', 'network_ops_count', 'memory_ops_count', 'io_ops_count', 'file_ops_unique', 'process_ops_unique', 'network_ops_unique', 'memory_ops_unique', 'io_ops_unique', 'file_ops_ratio', 'process_ops_ratio', 'network_ops_ratio', 'memory_ops_ratio', 'io_ops_ratio', 'first_execve_pos', 'first_connect_pos', 'first_socket_pos', 'first_write_pos', 'first_unlink_pos', 'first_rename_pos', 'first_mprotect_pos', 'first_clone_pos', 'has_execve', 'has_connect', 'has_socket', 'has_write', 'has_unlink', 'has_rename', 'has_chmod', 'has_mprotect', 'has_process_spawn_and_network', 'has_file_write_and_network', 'count_open', 'ratio_open', 'count_openat', 'ratio_openat', 'count_read', 'ratio_read', 'count_write', 'ratio_write', 'count_close', 'ratio_close', 'count_stat

,sample_id,label,trace_file_count,total_syscalls,unique_syscalls,failed_syscalls,failed_syscall_ratio,top1_syscall_ratio,top3_syscall_ratio,trace_entropy,...,early50_has_execve,early50_has_connect,early50_has_write,early100_unique,early100_file_count,early100_process_count,early100_network_count,early100_has_execve,early100_has_connect,early100_has_write
0,0x-smart-contracts-99.99.99,1,27,9555,45,2886,0.302041,0.485610,0.717111,2.644467,...,1,0,0,8,72,6,0,1,0,0
1,0xzyo-confutionrce-1.0.0,1,11,8266,32,2771,0.335229,0.560005,0.793612,2.213814,...,1,0,0,8,72,6,0,1,0,0
2,1231dai-1.0.0,1,19,8965,38,2826,0.315226,0.517568,0.744785,2.446209,...,1,0,0,8,72,6,0,1,0,0
3,1kzr-1.0.0,1,11,8261,32,2771,0.335432,0.560344,0.794093,2.211943,...,1,0,0,8,72,6,0,1,0,0
4,1password-sdk-examples-1.0.0,1,23,9158,44,2830,0.309019,0.506879,0.732256,2.520383,...,1,0,0,8,72,6,0,1,0,0


In [3]:
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum().sort_values(ascending=False).head(20))

sample_id                 object
label                      int64
trace_file_count           int64
total_syscalls             int64
unique_syscalls            int64
                           ...  
early100_process_count     int64
early100_network_count     int64
early100_has_execve        int64
early100_has_connect       int64
early100_has_write         int64
Length: 132, dtype: object

Missing values:
 sample_id               0
label                   0
trace_file_count        0
total_syscalls          0
unique_syscalls         0
failed_syscalls         0
failed_syscall_ratio    0
top1_syscall_ratio      0
top3_syscall_ratio      0
trace_entropy           0
file_ops_count          0
process_ops_count       0
network_ops_count       0
memory_ops_count        0
io_ops_count            0
file_ops_unique         0
process_ops_unique      0
network_ops_unique      0
memory_ops_unique       0
io_ops_unique           0
dtype: int64


In [4]:
non_numeric_cols = df.select_dtypes(exclude=['number']).columns.tolist()
print("Non-numeric columns:", non_numeric_cols)

Non-numeric columns: ['sample_id']


In [5]:
X = df.drop(columns=['sample_id', 'label'])
y = df['label']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (96, 130)
y shape: (96,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain label counts:\n", y_train.value_counts())
print("\nTest label counts:\n", y_test.value_counts())

Train shape: (76, 130)
Test shape: (20, 130)

Train label counts:
 label
0    39
1    37
Name: count, dtype: int64

Test label counts:
 label
1    10
0    10
Name: count, dtype: int64


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.7

Confusion Matrix:
 [[8 2]
 [4 6]]

Classification Report:

              precision    recall  f1-score   support

           0       0.67      0.80      0.73        10
           1       0.75      0.60      0.67        10

    accuracy                           0.70        20
   macro avg       0.71      0.70      0.70        20
weighted avg       0.71      0.70      0.70        20



In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

rf_cv = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_scores = cross_val_score(rf_cv, X, y, cv=cv, scoring='accuracy')
f1_scores = cross_val_score(rf_cv, X, y, cv=cv, scoring='f1')
precision_scores = cross_val_score(rf_cv, X, y, cv=cv, scoring='precision')
recall_scores = cross_val_score(rf_cv, X, y, cv=cv, scoring='recall')

print("CV Accuracy scores:", acc_scores)
print("Mean CV Accuracy:", acc_scores.mean())

print("\nCV F1 scores:", f1_scores)
print("Mean CV F1:", f1_scores.mean())

print("\nCV Precision scores:", precision_scores)
print("Mean CV Precision:", precision_scores.mean())

print("\nCV Recall scores:", recall_scores)
print("Mean CV Recall:", recall_scores.mean())

CV Accuracy scores: [0.85       0.73684211 0.78947368 0.84210526 0.84210526]
Mean CV Accuracy: 0.8121052631578948

CV F1 scores: [0.85714286 0.66666667 0.77777778 0.85714286 0.82352941]
Mean CV F1: 0.796451914098973

CV Precision scores: [0.81818182 1.         0.77777778 0.75       0.875     ]
Mean CV Precision: 0.8441919191919192

CV Recall scores: [0.9        0.5        0.77777778 1.         0.77777778]
Mean CV Recall: 0.7911111111111111


In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

gb = GradientBoostingClassifier(random_state=42)

gb_acc = cross_val_score(gb, X, y, cv=cv, scoring='accuracy')
gb_f1 = cross_val_score(gb, X, y, cv=cv, scoring='f1')
gb_precision = cross_val_score(gb, X, y, cv=cv, scoring='precision')
gb_recall = cross_val_score(gb, X, y, cv=cv, scoring='recall')

print("GB CV Accuracy:", gb_acc)
print("Mean GB Accuracy:", gb_acc.mean())

print("\nGB CV F1:", gb_f1)
print("Mean GB F1:", gb_f1.mean())

print("\nGB CV Precision:", gb_precision)
print("Mean GB Precision:", gb_precision.mean())

print("\nGB CV Recall:", gb_recall)
print("Mean GB Recall:", gb_recall.mean())

GB CV Accuracy: [0.9        0.84210526 0.73684211 0.84210526 0.73684211]
Mean GB Accuracy: 0.811578947368421

GB CV F1: [0.88888889 0.82352941 0.70588235 0.85714286 0.66666667]
Mean GB F1: 0.788422035480859

GB CV Precision: [1.         1.         0.75       0.75       0.83333333]
Mean GB Precision: 0.8666666666666666

GB CV Recall: [0.8        0.7        0.66666667 1.         0.55555556]
Mean GB Recall: 0.7444444444444445


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

svm_acc = cross_val_score(svm_model, X, y, cv=cv, scoring='accuracy')
svm_f1 = cross_val_score(svm_model, X, y, cv=cv, scoring='f1')
svm_precision = cross_val_score(svm_model, X, y, cv=cv, scoring='precision')
svm_recall = cross_val_score(svm_model, X, y, cv=cv, scoring='recall')

print("SVM CV Accuracy:", svm_acc)
print("Mean SVM Accuracy:", svm_acc.mean())

print("\nSVM CV F1:", svm_f1)
print("Mean SVM F1:", svm_f1.mean())

print("\nSVM CV Precision:", svm_precision)
print("Mean SVM Precision:", svm_precision.mean())

print("\nSVM CV Recall:", svm_recall)
print("Mean SVM Recall:", svm_recall.mean())

SVM CV Accuracy: [0.85       0.73684211 0.84210526 0.84210526 0.78947368]
Mean SVM Accuracy: 0.8121052631578948

SVM CV F1: [0.85714286 0.66666667 0.84210526 0.85714286 0.77777778]
Mean SVM F1: 0.8001670843776107

SVM CV Precision: [0.81818182 1.         0.8        0.75       0.77777778]
Mean SVM Precision: 0.8291919191919191

SVM CV Recall: [0.9        0.5        0.88888889 1.         0.77777778]
Mean SVM Recall: 0.8133333333333332


In [11]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import pandas as pd

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),
    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
    ])
}

results = []

for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    f1 = cross_val_score(model, X, y, cv=cv, scoring='f1')
    precision = cross_val_score(model, X, y, cv=cv, scoring='precision')
    recall = cross_val_score(model, X, y, cv=cv, scoring='recall')

    results.append({
        "Model": name,
        "Accuracy Mean": acc.mean(),
        "F1 Mean": f1.mean(),
        "Precision Mean": precision.mean(),
        "Recall Mean": recall.mean()
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Mean", ascending=False).reset_index(drop=True)

print(results_df)

               Model  Accuracy Mean   F1 Mean  Precision Mean  Recall Mean
0                SVM       0.812105  0.800167        0.829192     0.813333
1      Random Forest       0.812105  0.796452        0.844192     0.791111
2  Gradient Boosting       0.811579  0.788422        0.866667     0.744444


In [12]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X, y)

feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

feature_importance_df.head(15)

,Feature,Importance
0,first_socket_pos,0.108091
1,ratio_close,0.070781
2,first_write_pos,0.061506
3,total_syscalls,0.058220
4,ratio_openat,0.039526
5,count_write,0.033432
6,io_ops_count,0.030538
7,ratio_ioctl,0.030046
8,failed_syscall_ratio,0.029303
9,count_read,0.026902


In [13]:
feature_importance_df.head(10)

,Feature,Importance
0,first_socket_pos,0.108091
1,ratio_close,0.070781
2,first_write_pos,0.061506
3,total_syscalls,0.058220
4,ratio_openat,0.039526
5,count_write,0.033432
6,io_ops_count,0.030538
7,ratio_ioctl,0.030046
8,failed_syscall_ratio,0.029303
9,count_read,0.026902
